A **token** is a chunk of text the model actually "sees" — usually a **subword piece**, not a whole word and not a single character.
- Common English words like `"the"` or `"is"` are frequent enough to get **one whole token** to themselves
- Longer or rarer words get **split into pieces** — `"tokenization"` might become `"token"` + `"ization"`, two tokens for one word
- Digits and unusual strings often split heavily — `"BJ2019FD7717"` is far more than 1 token, even though it reads like one "word"
- This is why Claude can handle words it's **never seen before** — it stitches together pieces it does recognize, instead of failing outright

Anthropic gives you a **free, separate endpoint** — `client.messages.count_tokens()` — that tells you exactly how many tokens a request would use, **without generating anything**.
- It accepts the **same shape** as a real `messages.create()` call — same `model`, `system`, `messages` — so you're testing the exact request you're about to send
- It costs **nothing**, and has its **own** rate limit, separate from your generation limit
- It only tells you the **input** side — `output_tokens` can't be known in advance, since that depends on what the model decides to generate
- Useful for three things: staying under a model's **context window**, predicting **cost** before you spend it, and deciding if a prompt needs trimming

**Input tokens** and **output tokens** are billed at different rates — output runs about **5x more expensive** than input, on every Claude tier.
- **Input tokens** are processed all at once, in **parallel** — the model reads your whole prompt in a single pass
- **Output tokens** are generated one at a time, **sequentially** — each new token depends on everything generated before it, so generation can't be batched the same way reading can
- That sequential dependency is what the higher output price is reflecting
- Practical effect on your script: keeping `max_tokens` small isn't just about avoiding truncation — every output token is the **expensive** kind

Code-mixed Hinglish tends to need **more tokens per character** than clean English — the same underlying gap your MuRIL pipeline hit, just showing up as a different symptom.
- Claude's tokenizer vocabulary was shaped mostly by **huge volumes of standard English** — common English words earned efficient, dedicated token slots
- Romanized Hindi words (`"raha"`, `"hoon"`, `"jaldi"`) weren't part of that same frequency pattern, so they're more likely to get **split into multiple pieces** instead of landing one clean token
- It's not a hard failure like MuRIL's **`[UNK]`** — Claude never refuses to tokenize anything — but it's a **quieter cost**: the same idea in Hinglish reliably eats more tokens than in English
- The script below measures this directly: tokens-per-character on a real Hinglish phrase vs. its English equivalent

Once you have `input_tokens` and `output_tokens` from a real response, **cost** is just two multiplications and an addition — no guessing required.
- **Cost = (input_tokens ÷ 1,000,000) × input_price + (output_tokens ÷ 1,000,000) × output_price**
- For Haiku 4.5: **$1** per million input tokens, **$5** per million output tokens
- A typical one-email classification call (~190 input, ~50 output tokens) costs roughly **$0.0005**
- This is what turns "we have 2,500 emails" into a real number **before** running the batch — multiply per-call cost by row count instead of finding out after the bill arrives

In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic
import anthropic

load_dotenv()  # reads .env in the current working directory
api_key = os.getenv("anthropic_api_key")
client = anthropic.Anthropic(api_key=api_key)

MODEL_ID = "claude-haiku-4-5-20251001"
 
# Pricing per million tokens (Haiku 4.5) — update if you're on a different model
INPUT_PRICE_PER_MTOK = 1.0
OUTPUT_PRICE_PER_MTOK = 5.0

Free endpoint — same shape as messages.create(), but only returns how many input tokens the request WOULD use. No generation happens, no cost is incurred.

In [2]:
def count_tokens(text: str, system: str = None) -> int:

    kwargs = {
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": text}],
    }
    if system:
        kwargs["system"] = system
    response = client.messages.count_tokens(**kwargs)
    return response.input_tokens

Probe for token boundaries by growing `text` one character at a time. Returns the character indices where a NEW token starts (the very first character always trivially starts the first token, so it's excluded).

In [3]:
def find_token_boundaries(text: str) -> list:

    boundaries = []
    prev_count = count_tokens(text[0])
    for i in range(2, len(text) + 1):
        count = count_tokens(text[:i])
        if count > prev_count:
            boundaries.append(i - 1)
        prev_count = count
    return boundaries

Insert a pipe at every detected token-start boundary, for a readable view.

In [4]:
def visualize_boundaries(text: str, boundaries: list) -> str:
    
    result = []
    for i, ch in enumerate(text):
        if i in boundaries:
            result.append("|")
        result.append(ch)
    return "".join(result)

In [5]:
def calculate_cost(input_tokens: int, output_tokens: int) -> float:
    return (
        (input_tokens / 1_000_000) * INPUT_PRICE_PER_MTOK
        + (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MTOK
    )
 

In [6]:
SAMPLE_SYSTEM = (
        "You classify customer emails for Bajaj Finance into one of three "
        "categories: FD, Non-FD, Multiple Category."
    )

SAMPLE_EMAIL = (
        "Hello ji, Yeh second baar likh raha hoon. My funds with your company "
        "is stuck. The period was over in December but nothing came to my "
        "bank. Please check BJ2019FD7717. Jaldi kuch karo please. Thank you. "
        "Shobha Chopra | 9686667744"
    )

In [7]:
# ------------------------------------------------------------------
# 1. Counting tokens BEFORE sending — real email, real system prompt
# ------------------------------------------------------------------
print("=" * 70)
print("1. Token count BEFORE sending anything to the model")
print("=" * 70)
n_tokens = count_tokens(SAMPLE_EMAIL, system=SAMPLE_SYSTEM)
print(f"Email length    : {len(SAMPLE_EMAIL)} characters")
print(f"Input tokens    : {n_tokens}  (system prompt + email combined)")

1. Token count BEFORE sending anything to the model
Email length    : 228 characters
Input tokens    : 106  (system prompt + email combined)


In [8]:
# ------------------------------------------------------------------
# 2. Visualizing token boundaries — Hinglish vs. equivalent English
#    Kept SHORT on purpose: one API call per character.
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("2. Token boundaries — Hinglish phrase vs. English equivalent")
print("=" * 70)

hinglish_phrase = "Yeh second baar likh raha hoon"
english_phrase = "This is the second time writing"

hin_boundaries = find_token_boundaries(hinglish_phrase)
eng_boundaries = find_token_boundaries(english_phrase)

hin_tokens = len(hin_boundaries) + 1
eng_tokens = len(eng_boundaries) + 1

print(f"\nHinglish : {visualize_boundaries(hinglish_phrase, hin_boundaries)}")
print(f"  -> {hin_tokens} tokens across {len(hinglish_phrase)} characters")

print(f"\nEnglish  : {visualize_boundaries(english_phrase, eng_boundaries)}")
print(f"  -> {eng_tokens} tokens across {len(english_phrase)} characters")

print(f"\nTokens per character -> Hinglish: {hin_tokens / len(hinglish_phrase):.3f}"
        f"  |  English: {eng_tokens / len(english_phrase):.3f}")


2. Token boundaries — Hinglish phrase vs. English equivalent

Hinglish : Y|e|h| sec|ond| ba|ar| li|kh| ra|ha| ho|on
  -> 13 tokens across 30 characters

English  : T|h|is| is| the| sec|ond| time| w|riting
  -> 10 tokens across 31 characters

Tokens per character -> Hinglish: 0.433  |  English: 0.323


In [9]:
# ------------------------------------------------------------------
# 3. A real call — input vs output tokens, and the actual cost
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("3. A real call — input vs output tokens, and what it actually cost")
print("=" * 70)

response = client.messages.create(
    model=MODEL_ID,
    max_tokens=150,
    system=SAMPLE_SYSTEM,
    messages=[{"role": "user", "content": SAMPLE_EMAIL}],
)

input_tokens = response.usage.input_tokens
output_tokens = response.usage.output_tokens
cost = calculate_cost(input_tokens, output_tokens)

print(f"input_tokens  : {input_tokens}   (priced at ${INPUT_PRICE_PER_MTOK}/MTok)")
print(f"output_tokens : {output_tokens}   (priced at ${OUTPUT_PRICE_PER_MTOK}/MTok -> 5x input)")
print(f"Cost of this one call          : ${cost:.6f}")
print(f"Projected cost for 2,500 emails : ${cost * 2500:.2f}")


3. A real call — input vs output tokens, and what it actually cost
input_tokens  : 106   (priced at $1.0/MTok)
output_tokens : 131   (priced at $5.0/MTok -> 5x input)
Cost of this one call          : $0.000761
Projected cost for 2,500 emails : $1.90
